In [ ]:
!pip install rdkit
!pip install Bio
!pip install cudaq

In [ ]:
import os, math
import numpy as np
import networkx as nx
from rdkit import Chem, RDConfig
from rdkit.Chem import AllChem, ChemicalFeatures, rdchem, Descriptors, Lipinski, Crippen
from Bio.PDB import PDBParser, NeighborSearch
import cudaq
from cudaq import spin
import itertools
#Dont run more than once
# binding_scores, ligand_smiles_list = [], []


def rdkit_ligand_features(mol, confId=0, families=None):
    fdef = os.path.join(RDConfig.RDDataDir, "BaseFeatures.fdef")
    factory = ChemicalFeatures.BuildFeatureFactory(fdef)
    feats = []
    for f in factory.GetFeaturesForMol(mol, confId=confId):
        fam = f.GetFamily()
        if (families is None) or (fam in families):
            feats.append({
                "id": ("L", len(feats)),
                "type": fam,
                "pos": np.array([f.GetPos().x, f.GetPos().y, f.GetPos().z])
            })
    return feats

def protein_features_from_pdb(pdb_path, site_center=None, site_radius=8.0):
    """Return receptor feature points {id,type,pos} within site sphere."""
    structure = PDBParser(QUIET=True).get_structure("prot", pdb_path)
    atoms = [a for a in structure.get_atoms() if a.element != "H"]

    if site_center is None:
        coords = np.array([a.coord for a in atoms])
        site_center = coords.mean(0)

    feats = []
    for a in atoms:
        p = a.coord
        if np.linalg.norm(p - site_center) > site_radius:
            continue
        res = a.get_parent()
        resname = res.get_resname().strip()
        aname = a.get_name().strip()

        if aname == "O":
            feats.append({"id": ("R", len(feats)), "type": "Acceptor", "pos": p.copy()})


        if aname in ("N","NE","NE2","ND2","NZ"):
            feats.append({"id": ("R", len(feats)), "type": "Donor", "pos": p.copy()})

        if resname in ("LYS","ARG") and aname in ("NZ","CZ","NE","NH1","NH2"):
            feats.append({"id": ("R", len(feats)), "type": "PosIonizable", "pos": p.copy()})
        if resname in ("ASP","GLU") and aname.startswith("O"):
            feats.append({"id": ("R", len(feats)), "type": "NegIonizable", "pos": p.copy()})

        aromatic_res = {"PHE":{"CG","CD1","CD2","CE1","CE2","CZ"},
                        "TYR":{"CG","CD1","CD2","CE1","CE2","CZ"},
                        "TRP":{"CD2","CE2","CE3","CZ2","CZ3","CH2"},
                        "HIS":{"CG","ND1","CD2","CE1","NE2"}}
        if resname in aromatic_res and aname in aromatic_res[resname]:
            pass

    return feats

COMPLEMENT = {
    ("Donor","Acceptor"), ("Acceptor","Donor"),
    ("PosIonizable","NegIonizable"), ("NegIonizable","PosIonizable"),
    ("Aromatic","Aromatic"),
    ("Hydrophobe","Hydrophobe"),
}

def in_contact_window(tL, tR, dist):
    if {tL,tR}=={"Donor","Acceptor"}:      return 1.6 <= dist <= 3.3
    if {tL,tR}=={"PosIonizable","NegIonizable"}: return 2.0 <= dist <= 5.0
    if tL==tR=="Aromatic":                 return 3.5 <= dist <= 6.0
    if tL==tR=="Hydrophobe":               return 3.0 <= dist <= 6.5
    return False

def build_BIG(lig_feats, rec_feats, eps_pair=0.75, max_candidates_per_lig=20):
    nodes = []
    for i, lf in enumerate(lig_feats):
        candidates = []
        for j, rf in enumerate(rec_feats):
            if (lf["type"], rf["type"]) not in COMPLEMENT:
                continue
            d = np.linalg.norm(lf["pos"] - rf["pos"])
            if in_contact_window(lf["type"], rf["type"], d):
                candidates.append((d, i, j))
        candidates.sort()
        for _, i2, j in candidates[:max_candidates_per_lig]:
            nodes.append((i2, j))

    G = nx.Graph()
    G.add_nodes_from(nodes)

    for a in range(len(nodes)):
        i, j = nodes[a]
        xi, yj = lig_feats[i]["pos"], rec_feats[j]["pos"]
        for b in range(a+1, len(nodes)):
            k, m = nodes[b]
            if (i==k) or (j==m):
                continue
            xk, ym = lig_feats[k]["pos"], rec_feats[m]["pos"]
            if abs(np.linalg.norm(xi - xk) - np.linalg.norm(yj - ym)) <= eps_pair:
                G.add_edge((i,j), (k,m))
    return G

def kabsch(P, Q):
    Pc = P.mean(0); Qc = Q.mean(0)
    X = P - Pc; Y = Q - Qc
    H = X.T @ Y
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1,:] *= -1
        R = Vt.T @ U.T
    t = Qc - R @ Pc
    return R, t

def pose_from_clique(clique, lig_feats, rec_feats):
    P = np.array([lig_feats[i]["pos"] for (i,_) in clique])
    Q = np.array([rec_feats[j]["pos"] for (_,j) in clique])
    R, t = kabsch(P, Q)
    rms = np.sqrt(((Q - (P @ R.T + t))**2).sum(axis=1).mean())
    return R, t, rms

# Use the Methods to Create a Binding-Interaction Graph Between Ligand and Protein

In [ ]:
# Obtain SMILES String of ligand of interest from database
smiles = 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4' # Replace every time
lig = Chem.AddHs(Chem.MolFromSmiles(smiles))
AllChem.EmbedMolecule(lig, AllChem.ETKDG())
AllChem.MMFFOptimizeMolecule(lig)
lig_feats = rdkit_ligand_features(lig, confId=0, families=None)

# Obtain AlphaFold multimer result as a single PDB file of atomic coordinates
pdb_path = '/content/unrelaxed_model_1_pred_0.pdb' # Replace

# Identify Location of Binding site of interest (It's 3D coordinate in the AlphaFold-returned PDB)
binding_site_center = np.array([0, 0, 0]) # Replace
binding_site_radius = 8.0 # Defines a sphere with this radius around the binding center and collects pharamcophores in this sphere only
rec_feats = protein_features_from_pdb(pdb_path, site_center=binding_site_center, site_radius=binding_site_radius)

# Build the BIG
G = build_BIG(lig_feats, rec_feats, eps_pair=0.6, max_candidates_per_lig=15)

In [ ]:

node2idx = {n: k for k, n in enumerate(G.nodes())}
idx2node = {k: n for n, k in node2idx.items()}

nodes = list(idx2node.keys())
qubit_num = len(nodes)


edges = [(node2idx[u], node2idx[v]) for u, v in G.edges()]

non_edges = [
    (u, v)
    for u, v in itertools.combinations(nodes, 2)
    if (u, v) not in edges and (v, u) not in edges
]

weights = []
for k in nodes:
    iL, jR = idx2node[k]
    d = np.linalg.norm(lig_feats[iL]["pos"] - rec_feats[jR]["pos"])
    weights.append(1.0 / d)

penalty = 1.2 * max(weights)            # rule‑of‑thumb ≥ 1.2 × max weight
num_layers = 3#Chosen experimentally


In [ ]:

# # BIG 1 (Example from Nvidia)

# nodes = [0, 1, 2, 3, 4, 5] #Depends on number of ph4s
# qubit_num = len(nodes)
# edges = [[0, 1], [0, 2], [0, 4], [0, 5], [1, 2], [1, 3], [1, 5], [2, 3], [2, 4],
#          [3, 4], [3, 5], [4, 5]] #from checking every node pair against clashes
# non_edges = [
#     [u, v] for u in nodes for v in nodes if u < v and [u, v] not in edges
# ]

# print('Edges: ', edges)
# print('Non-Edges: ', non_edges)

# weights = [0.6686, 0.6686, 0.6686, 0.1453, 0.1453, 0.1453]
# penalty = 6.0
# #Set by user, penalizes edges of the graph with no interactions
# #rule of thumb is P ≳ 1.2 × (max weight)
# num_layers = 3#Chosen experimentally

In [ ]:
# Hamiltonian
def ham_clique(penalty, nodes, weights, non_edges) -> cudaq.SpinOperator:

    spin_ham = 0
    for wt, node in zip(weights, nodes):
        #print(wt,node)
        spin_ham += 0.5 * wt * spin.z(node)
        spin_ham -= 0.5 * wt * spin.i(node)

    for non_edge in non_edges:
        u, v = (non_edge[0], non_edge[1])
        #print(u,v)
        spin_ham += penalty / 4.0 * (spin.z(u) * spin.z(v) - spin.z(u) -
                                     spin.z(v) + spin.i(u) * spin.i(v))

    return spin_ham

In [ ]:
def term_coefficients(ham: cudaq.SpinOperator) -> list[complex]:
    result = []
    for term in ham:
        result.append(term.evaluate_coefficient())
    return result

    # Get corresponding Pauli wordss


def term_words(ham: cudaq.SpinOperator) -> list[str]:
    # Our kernel uses these words to apply exp_pauli to the entire state.

    result = []
    for term in ham:
        result.append(term.get_pauli_word(qubit_num))
    return result


ham = ham_clique(penalty, nodes, weights, non_edges)
print(ham)

coef = term_coefficients(ham)
words = term_words(ham)

print(term_coefficients(ham))
print(term_words(ham))

In [ ]:
@cudaq.kernel
def dc_qaoa(qubit_num:int, num_layers:int, thetas:list[float],\
    coef:list[complex], words:list[cudaq.pauli_word]):

    qubits = cudaq.qvector(qubit_num)

    h(qubits)

    count = 0
    for p in range(num_layers):

        for i in range(len(coef)):
            exp_pauli(thetas[count] * coef[i].real, qubits, words[i])
            count += 1

        for j in range(qubit_num):
            rx(thetas[count], qubits[j])
            count += 1

        #Comment out this for loop for conventional QAOA
        for k in range(qubit_num):
            ry(thetas[count], qubits[k])
            count += 1

In [ ]:
# Specify the optimizer and its initial parameters.
optimizer = cudaq.optimizers.NelderMead()

#Specify random seeds
np.random.seed(13)
cudaq.set_random_seed(13)

# if dc_qaoa used
parameter_count = (2 * qubit_num + len(coef)) * num_layers

# if qaoa used
# parameter_count=(qubit_num+len(coef))*num_layers

print('Total number of parameters: ', parameter_count)
optimizer.initial_parameters = np.random.uniform(-np.pi / 8, np.pi / 8,
                                                 parameter_count)
print("Initial parameters = ", optimizer.initial_parameters)

In [ ]:
cost_values = []


def objective(parameters):

    cost = cudaq.observe(dc_qaoa, ham, qubit_num, num_layers, parameters, coef,
                         words).expectation()
    cost_values.append(cost)
    return cost


# Optimize!
optimal_expectation, optimal_parameters = optimizer.optimize(
    dimensions=parameter_count, function=objective)

print('optimal_expectation =', optimal_expectation)
print('optimal_parameters =', optimal_parameters)

In [ ]:
shots = 200000

counts = cudaq.sample(dc_qaoa,
                      qubit_num,
                      num_layers,
                      optimal_parameters,
                      coef,
                      words,
                      shots_count=shots)
print(counts)

print('The MVWCP is given by the partition: ', counts.most_probable())

In [ ]:
binding_score = -optimal_expectation
ligand_smiles_list.append(smiles)
binding_scores.append(binding_score)
print(binding_scores)
print(set(ligand_smiles_list))

## Run the below AFTER all the ligands and their SMILES have been run

In [ ]:
!pip install admet_ai
!pip install torch
!pip install argparse

In [ ]:
import torch
from argparse import Namespace
from admet_ai import ADMETModel

In [ ]:
torch.serialization.add_safe_globals([Namespace])

model = ADMETModel()

admet_df = model.predict(smiles=ligand_smiles_list)

# Lipinski drug-likeness filter
def lipinski_ok(mol):
    return (Descriptors.MolWt(mol) < 500 and
            Crippen.MolLogP(mol)  < 5   and
            Lipinski.NumHDonors(mol)    <= 5 and
            Lipinski.NumHAcceptors(mol) <= 10)

lipinski_flags = [lipinski_ok(Chem.MolFromSmiles(smi)) for smi in ligand_smiles_list]
admet_df["lipinski_pass"] = lipinski_flags

# Simple ADMET filters
good_admet = (
    (admet_df["HIA_prob"]        > 0.8) &   # absorption
    (admet_df["BBB_permeable"]   < 0.5) &   # avoid CNS
    (admet_df["hERG_inhib_prob"] < 0.3) &   # cardiotox
    (admet_df["CYP3A4_inhib"]    < 0.5) &   # DDI risk
    (admet_df["lipinski_pass"])
)
admet_df["admet_pass"] = good_admet

# Combine with binding scores and sort
rank_df = (
    admet_df
        .assign(binding_score=binding_scores)
        .sort_values(["admet_pass", "binding_score"], ascending=[False, False])
        .reset_index(drop=True)
)

print("FINAL RANKED LIGANDS")
print(rank_df[["SMILES", "binding_score", "admet_pass"]].head(10))
